In [1]:
# import libraries
import re
import string
import heapq
import numpy as np
import pandas as pd
from tqdm import tqdm
from urllib.request import urlopen

# if you need to plot anything
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt

# Stopword dictionary
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk import download, wordnet
nltk.download('omw-1.4')
download('punkt')
download('stopwords')
download("wordnet")

# For stemming
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()


from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
cv = CountVectorizer()
tfidf = TfidfVectorizer()

# let's import ML algorithms for classification
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split as tts, KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score as ac, confusion_matrix as cm, ConfusionMatrixDisplay as CMD

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dvasiliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/dvasiliu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /home/dvasiliu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dvasiliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/dvasiliu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
# here we import a dataset that is not trivial
# the first example uses only 1000 observations
wine_data = pd.read_csv('https://github.com/dvasiliu/AML/blob/main/Data%20Sets/winemagdata130kv2.csv?raw=true',quoting=2)
wines = wine_data[["description","points"]]
# wines_subset = wines.sample(1000,random_state=1234).reset_index(drop=True)
# corpus = []

In [ ]:
wines.description[0]

In [ ]:
np.where(wines.points==95)

In [ ]:
wines.description[357]

In [ ]:
stop_words = set(stopwords.words('english')) # is read as a list -> a search within this list to discard found stopwords in the text -> VERY inefficient

In [ ]:
def fast_remove_stopwords(texts):

    stop_words = set(stopwords.words('english'))
    
    # Compile regex for efficiency (matches whole words only)
    pattern = r'\b(?:{})\b'.format('|'.join(stop_words))
    regex = re.compile(pattern, re.IGNORECASE)
    
    # Process all texts with a single function call
    return [regex.sub('', text) for text in texts]

In [ ]:
no_stop_words = fast_remove_stopwords(wines.description)

In [3]:
def fast_preprocess(texts, lemmatize=True):
    
    # Fast text preprocessing function that handles:
    # - Lowercasing
    # - Punctuation removal
    # - Stopword removal
    # - Lemmatization (optional)
    
    # Arguments:
    #     texts: List of strings or single string to process
    #     stop_words: Set of stopwords (if None, uses NLTK English stopwords)
    #     lemmatize: Whether to perform lemmatization
    
    # Returns:
    #     Processed texts (list or string, matching input type)
    
    # Handle single string input
    single_input = isinstance(texts, str)
    if single_input:
        texts = [texts]
    
    
    
    stop_words = set(stopwords.words('english'))
    
    # Create translator for punctuation removal (faster than regex for this task)
    translator = str.maketrans('', '', string.punctuation)
    
    # Initialize lemmatizer if needed
    lemmatizer = WordNetLemmatizer() if lemmatize else None
    
    processed_texts = []
    for text in texts:
        # Lowercase
        text = text.lower()
        
        # Remove punctuation
        text = text.translate(translator)
        
        # Tokenize (needed for both stopword removal and lemmatization)
        tokens = text.split()
        
        # Remove stopwords
        tokens = [token for token in tokens if token not in stop_words]
        
        # Lemmatize if requested
        if lemmatize:
            tokens = [lemmatizer.lemmatize(token) for token in tokens]
        
        # Rejoin into text
        processed_texts.append(' '.join(tokens))
    
    # Return in same format as input
    return processed_texts[0] if single_input else processed_texts

In [4]:
%time preprocessed_reviews = fast_preprocess(wines.description)

CPU times: user 9.98 s, sys: 172 ms, total: 10.2 s
Wall time: 10.2 s


In [ ]:
preprocessed_reviews[356]

In [ ]:
len(preprocessed_reviews)

In [17]:
# first we'll create an embedding by using the whole data
embedding = TfidfVectorizer()
x = embedding.fit_transform(preprocessed_reviews)

In [8]:
# now we move our attention to the dependent variable y
y = wines.points

In [ ]:
y.describe() # we can treat y as a continuous variable even if it's not actually continuous -> this allows you to use the MSE as a metric in this context

In [ ]:
# so we can use any regressor
# first, let's try ElasticNet
model = ElasticNet(alpha=0.01,l1_ratio=0.5,max_iter=1000)
model.fit(x,y)

In [ ]:
# SO, what is the MSE of predictions on the whole data?
np.mean((y-model.predict(x))**2)

We want to make an objective function with kFold cross-validations to find the best hyperparameters

In [5]:
def objective(h):
    model = ElasticNet(alpha=h[0],l1_ratio=h[1],max_iter=1000)
    pipe = Pipeline([['embedding',TfidfVectorizer()],['scale',MaxAbsScaler()],['enet',model]])
    kf = KFold(n_splits=5,shuffle=True)
    return np.mean(-cross_val_score(pipe,preprocessed_reviews,y,cv=kf,scoring='neg_mean_squared_error'))

In [8]:
%time objective([0.1,0.5])

CPU times: user 48.7 s, sys: 387 ms, total: 49.1 s
Wall time: 46.2 s


np.float64(9.219549861568108)

In [5]:
# we experiment with Support Vector Regression
from cuml import SVR

In [11]:
def objective_SVR(h):
    model = SVR(kernel='poly',epsilon=h[0],C=h[1],degree=1)
    pipe = Pipeline([['embedding',TfidfVectorizer()],['scale',MaxAbsScaler()],['SVR',model]])
    kf = KFold(n_splits=5,shuffle=True)
    return np.mean(-cross_val_score(pipe,preprocessed_reviews,y,cv=kf,scoring='neg_mean_squared_error'))

In [15]:
%time objective_SVR([3,10])

CPU times: user 13min 33s, sys: 46.7 s, total: 14min 20s
Wall time: 14min 21s


np.float64(3.2471772576709315)